In [ ]:
#서울시 응답소 사이트 정보 수집하기
#Step 1. 필요한 모듈과 라이브러리를 로딩합니다.
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import pandas as pd  
import random
import os   # 폴더 생성하기 위해 필요함
import pyautogui

#Step 2. 사용자에게 크롤링에 필요한 정보를 입력 받습니다.

query_txt = '서울시응답소'
cnt = int(input('1.크롤링 할 건수는 몇건입니까?: '))
page_cnt = math.ceil(cnt / 10)
print("크롤링 할 총 페이지 번호: ",page_cnt)
f_dir = input("2.결과 파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir=='' :
    f_dir ='c:\\py_temp\\'
    
print()

# 저장될 위치와 이름을 지정합니다
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)
os.makedirs(f_dir+s+'-'+query_txt)
os.chdir(f_dir+s+'-'+query_txt)
ff_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xlsx'

#Step 3. 크롬드라이버 설정 -> 응답소 사이트 접속 -> 메뉴클릭
s_time = time.time( )  # 시작시간 스탑워치 클릭

s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get('https://eungdapso.seoul.go.kr/')
driver.maximize_window()

#페이지가 다 열릴 때까지 2-5초 사이 기다리기
time.sleep(random.randrange(2,5))  

#시장에게 바랍니다 메뉴 클릭하기
driver.find_element(By.XPATH,'//*[@id="content"]/div[2]/a/div/div/div[1]/div[1]').click()

In [ ]:
# Step 4.게시물의 정보 수집

no2=[]             # 게시글 번호 컬럼
title2=[]          # 게시글 제목 컬럼
odate2=[]          # 게시글 공개일 컬럼
contents2=[]       # 게시글 상담내역 컬럼

no = 1    # 게시글 번호 카운트 변수

for x in range(1,page_cnt+1) :
    print("%s 페이지 내용 수집 시작합니다 =======================" %x)
        
    # 각 페이지에 있는 게시글 10건의 상세 내역 가져오기
    for i in range(1,11) :
        
        if no > cnt :
            break
        else :
            
            #txt 형식으로 저장할 파일 지정
            f = open(ff_name, 'a',encoding='UTF-8')           

            # 게시글 제목 클릭
            print('===== %s번째 게시물의 상세내역을 추출합니다=====' %no)
            driver.find_element(By.XPATH,'//*[@id="content"]/div[2]/div[4]/table/tbody/tr[%s]/td[2]/div/a' %i).click( )
                                          
                                          
            time.sleep(1)
                               
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
           
            # 1.번호 입력
            no2.append(no)
            
            # 2.제목 추출
            title = soup.find('h3','sclinkage_title').get_text()
            print('1.제목:' , title)
            f.write('1.제목:' + title + '\n' )
            title2.append(title)
            
            # 3.공개일 추출
            odate = soup.find('span','sclinkage_evalue').get_text()
            print('2.공개일:', odate)
            f.write('2.공개일:' + odate + '\n' )
            odate2.append(odate)
                       
            # 4.상담내역 추출
            contents = soup.find('div','restd_incont').get_text()
            print('3.상담내역:', contents , '\n')
            f.write('3.상담내역:' + contents +'\n')
            contents2.append(contents)
            f.close( )
            

            no += 1

            #이전 페이지로 돌아가기
            driver.back()

            time.sleep(2)
        
    time.sleep(1)      
    
    x += 1    
    try :
        driver.find_element(By.LINK_TEXT,'%s' %x).click() # 다음 페이지번호 클릭
    except :
        driver.find_element(By.LINK_TEXT,'다음 페이지로 이동').click()

#Lesson 3 - 수집된 정보를 다양한 형식의 파일로 저장하기
# Step 6. 수집된 정보를 다양한 형식의 파일로 저장하기 
seoul = pd.DataFrame()
seoul['번호'] = no2
seoul['제목'] = title2
seoul['공개일'] = odate2
seoul['상담내용']=contents2

# csv 형태로 저장하기
seoul.to_csv(fc_name,encoding="utf-8-sig",index=False)

# 엑셀 형태로 저장하기
seoul.to_excel(fx_name,index=False, engine='openpyxl')

#Step 7. 요약 정보 출력하기
e_time = time.time( )     # 검색이 종료된 시점의 timestamp 를 지정합니다
t_time = e_time - s_time

print("\n") 
print("=" *100)
print("총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("파일 저장 완료: txt 파일명 : %s " %ff_name)
print("파일 저장 완료: csv 파일명 : %s " %fc_name)
print("파일 저장 완료: xls 파일명 : %s " %fx_name)
print("=" *100)

driver.close( )

In [ ]:
i